In [0]:
# ============================================================
# NOTEBOOK 5 — SHAP Explainability
# Team AryaBit | HackBricks 2026
# Pillar: EXPLAIN — per-student, human-readable risk reasons
# ============================================================

import logging
import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.SHAP")

# ── 0. Install SHAP ──────────────────────────────────────────
%pip install shap -q
dbutils.library.restartPython()

In [0]:
# Restart Python to use the newly installed SHAP package
dbutils.library.restartPython()

In [0]:
# ── Re-run after restart ─────────────────────────────────────
import shap
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, ArrayType
)
import logging, warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.SHAP")

spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

CATALOG   = "workspace"
SCHEMA    = "default"
EXP_NAME  = "/AryaBit_HackBricks_2026"

FEATURES = [
    "Curricular_units_2nd_sem_approved",
    "Curricular_units_2nd_sem_grade",
    "Curricular_units_1st_sem_approved",
    "Curricular_units_1st_sem_grade",
    "Tuition_fees_up_to_date",
    "Scholarship_holder",
    "Age_at_enrollment",
    "Debtor",
    "overall_approval_rate",
    "avg_grade_overall",
    "grade_delta",
    "absenteeism_trend",
    "financial_stress_index",
]

FRIENDLY_NAMES = {
    "overall_approval_rate":              "Low course approval rate",
    "Curricular_units_2nd_sem_approved":  "Units approved (Sem 2)",
    "Curricular_units_1st_sem_approved":  "Units approved (Sem 1)",
    "Curricular_units_2nd_sem_grade":     "Grade (Sem 2)",
    "Curricular_units_1st_sem_grade":     "Grade (Sem 1)",
    "avg_grade_overall":                  "Overall average grade",
    "grade_delta":                        "Grade trajectory (Sem1→2)",
    "absenteeism_trend":                  "Approval trend (Sem1→2)",
    "financial_stress_index":             "Financial stress level",
    "Tuition_fees_up_to_date":            "Tuition fees overdue",
    "Scholarship_holder":                 "No scholarship support",
    "Debtor":                             "Outstanding debt",
    "Age_at_enrollment":                  "Age at enrollment",
}

In [0]:
# ── 1. Load Model ────────────────────────────────────────────
logger.info("Loading model from MLflow UC registry...")

client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(EXP_NAME)

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="metrics.roc_auc > 0.92",
    order_by=["metrics.roc_auc DESC"],
    max_results=1,
)
assert runs, "No qualifying run found — re-run Notebook 3"
best_run_id = runs[0].info.run_id

rf_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/random_forest_model")
logger.info(f"Model loaded — run {best_run_id} ✅")

In [0]:
# ── 2. Load Silver Data ──────────────────────────────────────
logger.info("Loading Silver table...")

silver_df = spark.table(f"{CATALOG}.{SCHEMA}.silver_student_dropout")
pdf = silver_df.select(
    F.monotonically_increasing_id().alias("_row_id"),
    *FEATURES
).toPandas()

X = pdf[FEATURES].fillna(0)
logger.info(f"Loaded {len(X)} records for SHAP analysis")

In [0]:
# ── 3. Compute SHAP Values (version-safe) ───────────────────
logger.info("Initializing SHAP TreeExplainer...")

# Extract the RandomForest model from the pipeline
rf_classifier = rf_model.named_steps['model']

# Get the exact 17 features the model was trained with
model_features = [
    "grade_delta", "absenteeism_trend", "financial_stress_index",
    "overall_approval_rate", "avg_grade_overall",
    "curricular_units_1st_sem_grade", "curricular_units_2nd_sem_grade",
    "curricular_units_1st_sem_approved", "curricular_units_2nd_sem_approved",
    "curricular_units_1st_sem_enrolled", "curricular_units_2nd_sem_enrolled",
    "admission_grade", "age_at_enrollment", "debtor",
    "tuition_fees_up_to_date", "scholarship_holder", "displaced"
]

# Reload X with correct feature names from silver table
silver_df_correct = spark.table(f"{CATALOG}.{SCHEMA}.silver_student_dropout")
X_correct = silver_df_correct.select(model_features).toPandas().fillna(0)

explainer = shap.TreeExplainer(rf_classifier)

logger.info(f"Computing SHAP values for {len(X_correct)} students (~60s)...")
raw_shap = explainer.shap_values(X_correct)

# Identify dropout class index
classes     = list(rf_classifier.classes_)
dropout_idx = classes.index(1) if 1 in classes else 0
logger.info(f"Classes: {classes} | Dropout index: {dropout_idx}")

# ── Handle ALL SHAP return formats ──────────────────────────
if isinstance(raw_shap, list):
    # Old API: list of (n_samples, n_features) arrays, one per class
    shap_dropout = np.array(raw_shap[dropout_idx])

elif hasattr(raw_shap, "values"):
    # New API: shap.Explanation object → .values is (n_samples, n_features, n_classes)
    shap_dropout = raw_shap.values[:, :, dropout_idx]

elif isinstance(raw_shap, np.ndarray) and raw_shap.ndim == 3:
    # Some versions: raw ndarray (n_samples, n_features, n_classes)
    shap_dropout = raw_shap[:, :, dropout_idx]

else:
    # Binary fallback — shape (n_samples, n_features)
    shap_dropout = np.array(raw_shap)

# ── Hard assertion — catch shape issues before they cascade ─
expected = (len(X_correct), len(model_features))
assert shap_dropout.shape == expected, (
    f"❌ SHAP shape mismatch: got {shap_dropout.shape}, expected {expected}\n"
    f"raw_shap type={type(raw_shap)}, "
    f"raw_shap shape={np.array(raw_shap).shape if not hasattr(raw_shap,'values') else raw_shap.values.shape}"
)

logger.info(f"✅ SHAP matrix shape confirmed: {shap_dropout.shape}")

In [0]:
# ── 4. Build Per-Student SHAP Table ─────────────────────────
logger.info("Building per-student SHAP explanations...")

# Create DataFrame with row IDs for X_correct
pdf_with_ids = X_correct.copy()
pdf_with_ids.insert(0, '_row_id', range(len(X_correct)))

records = []
for i in range(len(X_correct)):
    row_shap   = shap_dropout[i]        # SHAP values for this student
    row_values = X_correct.iloc[i]      # Actual feature values

    # Rank features by absolute SHAP contribution
    ranked_idx = np.argsort(np.abs(row_shap))[::-1]

    top_factors = []
    for j in ranked_idx[:3]:
        feat        = model_features[j]
        shap_val    = float(row_shap[j])
        actual_val  = float(row_values[feat])
        direction   = "↑ risk" if shap_val > 0 else "↓ risk"
        label       = FRIENDLY_NAMES.get(feat, feat)
        top_factors.append(f"{label}: {actual_val:.2f} ({direction})")

    records.append({
        "_row_id":           int(pdf_with_ids["_row_id"].iloc[i]),
        "shap_dropout_prob": float(np.sum(row_shap[row_shap > 0])),
        "top_factor_1":      top_factors[0] if len(top_factors) > 0 else None,
        "top_factor_2":      top_factors[1] if len(top_factors) > 1 else None,
        "top_factor_3":      top_factors[2] if len(top_factors) > 2 else None,
    })

shap_pdf = pd.DataFrame(records)
logger.info(f"SHAP explanations built for {len(shap_pdf)} students ✅")

In [0]:
# ── 5. Merge SHAP into Gold Table ────────────────────────────
logger.info("Merging SHAP explanations into Gold table...")

shap_sdf = spark.createDataFrame(shap_pdf)

gold_df = spark.table(f"{CATALOG}.{SCHEMA}.gold_at_risk_students")

# Gold table has student_id = STU_XXXX — reconstruct row index
# Add row index to gold for join (based on original Silver order)
gold_with_idx = gold_df.withColumn(
    "_row_id",
    F.regexp_extract("student_id", r"STU_(\d+)", 1).cast("long")
)

gold_explained = (
    gold_with_idx
    .join(shap_sdf, on="_row_id", how="left")
    .drop("_row_id")
)

# ── 6. Write Gold Explained Table ───────────────────────────
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gold_explained_students"

(
    gold_explained
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)
logger.info(f"Gold explained table written: {TARGET_TABLE} ✅")

In [0]:
# ── 7. Validation + Print ────────────────────────────────────
sample = spark.table(TARGET_TABLE).filter(
    F.col("intervention_tier") == "HIGH"
).orderBy(F.col("risk_score").desc()).limit(5)

print("\n" + "="*80)
print("SHAP EXPLAINABILITY — TOP 5 HIGH-RISK STUDENTS")
print("Every score is now a story. Not a black box.")
print("="*80)

sample.select(
    "student_id", "risk_score", "intervention_tier",
    "top_factor_1", "top_factor_2", "top_factor_3", "bias_flag"
).show(truncate=False)

# Global Feature Importance (for presentation slide)
mean_abs_shap = np.abs(shap_dropout).mean(axis=0)
importance_df = (
    pd.DataFrame({
        "feature":    model_features,
        "mean_shap":  mean_abs_shap,
        "label":      [FRIENDLY_NAMES.get(f, f) for f in model_features],
    })
    .sort_values("mean_shap", ascending=False)
    .reset_index(drop=True)
)

print("\n📊 GLOBAL FEATURE IMPORTANCE (SHAP-based):")
print(importance_df[["label", "mean_shap"]].to_string(index=False))

logger.info("Notebook 5 COMPLETE ✅")